## Vision Transformer (ViT)
Vision Transformer(ViT)는 이미지를 처리할 때 CNN 대신 Transformer 구조를 사용하는 딥러닝 모델이다.<br>
원래 Transformer는 자연어 처리(NLP)용 모델이었는데, 이를 이미지 처리에 적용한 것이 Vision Transformer이다.

https://discuss.pytorch.kr/t/vision-transformer-a-visual-guide-to-vision-transformers/4158

PyTorch로 Vision Transformer(ViT)를 처음부터 구현하는 학습용 노트북입니다. <br>
이미지를 패치로 나누고 Transformer Encoder로 특징을 학습한 뒤, FashionMNIST 의류 이미지 10클래스 분류를 학습·평가합니다(테스트 정확도 약 87%).

In [1]:
# ========== 하이퍼파라미터 설정 ==========

# (1) 데이터 및 입력 관련
BATCH_SIZE   = 128      # 한 번에 처리할 이미지 수
IMAGE_SIZE   = 32       # 입력 이미지 한 변의 크기 (32x32)
IN_CHANNELS  = 1        # 입력 채널 수 (FashionMNIST=1, RGB 이미지=3)

# (2) ViT 모델 구조 관련
PATCH_SIZE   = 16       # 이미지를 나눌 패치(조각) 크기
NUM_HEADS    = 3        # Multi-Head Attention의 헤드 개수
NUM_ENCODERS = 3        # Transformer Encoder 블록 반복 횟수
EMBED_DIM    = 72       # 패치 임베딩 벡터 차원
MLP_SIZE     = EMBED_DIM * 4    # Feed-Forward MLP 은닉층 차원 (embed_dim의 4배, 72*4=288)

# (3) 패치 개수 계산
NUM_PATCHES  = (IMAGE_SIZE//PATCH_SIZE) ** 2    # 패치 수 = (32//16)^2 = 4

# (4) 학습 관련
DROPOUT_RATE = 0.1      # 드롭아웃 비율 (과적합 방지)
NUM_CLASSES  = 10       # 분류 클래스 수 (FashionMNIST 10개 클래스)

In [2]:
# torchinfo: 모델 구조 및 파라미터 수 요약 출력용 패키지
%pip install torchinfo

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ========== Torch 라이브러리 import ==========
import torch
import torch.nn as nn
from torchinfo import summary           # 모델 구조·파라미터 수 요약
import torch.optim as optim             # 옵티마이저
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms  # 데이터셋 및 이미지 전처리
import matplotlib.pyplot as plt
import numpy as np

# GPU 사용 가능 시 cuda, 아니면 cpu
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [3]:
# ========== 패치 분할 방식 1: Unfold + Linear Projection ==========
# 이미지를 패치 단위로 펼친 뒤, 선형층으로 임베딩 차원으로 투영
class PatcherUnfold(nn.Module):
    def __init__(self):
        super().__init__()
        # (1) Unfold: 이미지를 PATCH_SIZE 크기의 패치로 잘라 1차원으로 펼침
        self.unfold = nn.Unfold(kernel_size=PATCH_SIZE, stride=PATCH_SIZE)
        # (2) Linear Projection: 패치 픽셀값 → EMBED_DIM 차원 벡터로 변환
        self.linear_projection = nn.Linear(in_features=IN_CHANNELS*PATCH_SIZE*PATCH_SIZE, 
                                           out_features=EMBED_DIM)

    def forward(self, x):  
        print(f'original\t: {x.size()}')
        
        x = self.unfold(x)                          # [B, C*P*P, NUM_PATCHES]
        print(f'after unfold\t: {x.size()}')
        
        x = x.permute(0, 2, 1)                      # [B, NUM_PATCHES, C*P*P] 형태로 변환
        print(f'after permute\t: {x.size()}')
        
        x = self.linear_projection(x)             # [B, NUM_PATCHES, EMBED_DIM]
        print(f'after lin proj\t: {x.size()}')
        
        return x


In [4]:
# PatcherUnfold 동작 확인 (더미 입력: 배치 128, 채널 1, 32x32)
patcher_unfold = PatcherUnfold()
x = torch.randn(128, 1, 32, 32)  #  테스트용 가짜(더미) 이미지를 생성
x = patcher_unfold(x)

original	: torch.Size([128, 1, 32, 32])
after unfold	: torch.Size([128, 256, 4])
after permute	: torch.Size([128, 4, 256])
after lin proj	: torch.Size([128, 4, 72])


In [5]:
# ========== 패치 분할 방식 2: Conv2d (실제 ViT에서 주로 사용) ==========
# stride=PATCH_SIZE인 컨볼루션(convolution)으로 패치 분할과 임베딩을 동시에 수행
class PatcherConv(nn.Module):
    def __init__(self):
        super().__init__()
        # kernel=stride=PATCH_SIZE → 겹치지 않는 패치 단위로 conv 수행
        self.conv = nn.Conv2d(in_channels=IN_CHANNELS, 
                              out_channels=EMBED_DIM, 
                              kernel_size=PATCH_SIZE, 
                              stride=PATCH_SIZE)
        # 공간 차원(H, W)을 1차원으로 펼침
        self.flatten = nn.Flatten(start_dim=2)
    
    def forward(self, x): # 뒤에서 FashionMNIST 모델 학습 시 사용하기 위해 print문을 모두 주석 처리 해놓고 사용 할것!!
        print(f'original\t\t: {x.size()}')
        
        x = self.conv(x)                            # [B, EMBED_DIM, H/P, W/P]
        print(f'after conv\t\t: {x.size()}')
        
        x = self.flatten(x)                         # [B, EMBED_DIM, NUM_PATCHES]
        print(f'after flatten\t\t: {x.size()}')
        
        x = x.permute(0, 2, 1)                      # [B, NUM_PATCHES, EMBED_DIM]
        print(f'after permute\t\t: {x.size()}')
        
        return x

In [6]:
# PatcherConv 동작 확인 (Unfold 방식과 동일한 출력 형태 [B, NUM_PATCHES, EMBED_DIM])
patcher_conv = PatcherConv()
x = torch.randn(128, 1, 32, 32)
x = patcher_conv(x)

original		: torch.Size([128, 1, 32, 32])
after conv		: torch.Size([128, 72, 2, 2])
after flatten		: torch.Size([128, 72, 4])
after permute		: torch.Size([128, 4, 72])


In [7]:
# ========== 위치 임베딩 + Class Token ==========
# Transformer는 순서 정보가 없으므로, 학습 가능한 위치 임베딩을 더함
# Class Token: 전체 이미지 표현을 담는 특수 토큰 (최종 분류에 사용)
class PosEmbedding(nn.Module):
    def __init__(self):
        super().__init__()
        # 분류용 [CLS] 토큰 (BERT/ViT와 동일한 개념)
        self.class_token = nn.Parameter(torch.randn(1, 1, EMBED_DIM))
        # 패치 + class token 위치별 임베딩 (NUM_PATCHES + 1개)
        self.pos_embedding = nn.Parameter(torch.randn(1, NUM_PATCHES + 1, EMBED_DIM))
        self.dropout = nn.Dropout(p=DROPOUT_RATE)

    def forward(self, x):
        # x: [B, NUM_PATCHES, EMBED_DIM]
        B = x.size(0)
        class_token = self.class_token.expand(B, -1, -1)  # 배치 크기에 맞게 확장 → [B, 1, D]
        x = torch.cat([class_token, x], dim=1)            # class token을 맨 앞에 붙임 → [B, N+1, D]
        x = x + self.pos_embedding                        # 위치 임베딩 더하기 (브로드캐스팅)
        x = self.dropout(x)

        return x


In [8]:
# PosEmbedding 동작 확인 (패치 시퀀스에 class token + 위치 정보 추가)
pos_embedding = PosEmbedding()
x = pos_embedding(x)

In [9]:
# ========== Transformer Encoder 블록 ==========
# Pre-Norm 구조: LayerNorm → Attention/MLP → Residual Connection
class TransformerEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.norm_0 = nn.LayerNorm(EMBED_DIM)    # Attention 전 정규화
        
        # Multi-Head Self-Attention: 패치 간 관계 학습
        self.multihead_attention = nn.MultiheadAttention(EMBED_DIM, 
                                                         num_heads=NUM_HEADS, 
                                                         batch_first=True, 
                                                         dropout=DROPOUT_RATE)
        
        self.norm_1 = nn.LayerNorm(EMBED_DIM)    # MLP 전 정규화
        
        # Feed-Forward MLP (2층 선형 + GELU 활성화)
        self.mlp = nn.Sequential(
            nn.Linear(in_features=EMBED_DIM, out_features=MLP_SIZE),   # 확장
            nn.GELU(), 
            nn.Dropout(p=DROPOUT_RATE), 
            nn.Linear(in_features=MLP_SIZE, out_features=EMBED_DIM),   # 원래 차원으로 축소
            nn.Dropout(p=DROPOUT_RATE)
        )

    def forward(self, x):
        # --- Attention 서브레이어 (Residual Block 1) ---
        residual = x                              # 잔차 연결용 원본 저장
        
        x = self.norm_0(x)
        x = self.multihead_attention(x, x, x)[0]  # Self-Attention (Q=K=V=x)
        x = x + residual                          # 잔차 연결
        
        # --- MLP 서브레이어 (Residual Block 2) ---
        residual = x
        
        x = self.norm_1(x)
        x = self.mlp(x)
        x = x + residual                          # 잔차 연결
        
        return x

In [10]:
# TransformerEncoder 동작 확인
transformer_encoder = TransformerEncoder()
x = transformer_encoder(x)

In [11]:
# ========== 분류 헤드 (MLP Head) ==========
# Class Token의 최종 표현을 NUM_CLASSES개 로짓(logit)으로 변환
class MLPHead(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.norm = nn.LayerNorm(EMBED_DIM)
        self.linear_0 = nn.Linear(in_features=EMBED_DIM, 
                                  out_features=EMBED_DIM)
        self.gelu = nn.GELU()
        # 최종 출력층: 클래스 수만큼의 로짓
        self.linear_1 = nn.Linear(in_features=EMBED_DIM, 
                                  out_features=NUM_CLASSES)
        
    def forward(self, x):
        x = self.norm(x)
        x = self.linear_0(x)
        x = self.gelu(x)
        x = self.linear_1(x)                      # [B, NUM_CLASSES]
        
        return x

In [12]:
# Class Token(인덱스 0)만 추출하여 분류 헤드에 입력
x = x[:, 0]    # [B, EMBED_DIM] — 전체 이미지를 대표하는 벡터
mlp_head = MLPHead()
x = mlp_head(x)

In [13]:
# ========== Vision Transformer (ViT) 전체 모델 ==========
class ViT(nn.Module):
    def __init__(self):
        super().__init__()
    
        # (1) 패치 분할 + 임베딩 (Conv 방식 사용, Unfold 방식도 가능)
        #self.patcher = PatcherUnfold()
        self.patcher = PatcherConv()
        # (2) Class Token + 위치 임베딩
        self.pos_embedding = PosEmbedding()
        # (3) Transformer Encoder를 NUM_ENCODERS번 쌓음
        self.transformer_encoders = nn.Sequential(
            *[TransformerEncoder() for _ in range(NUM_ENCODERS)]
            )
        # (4) 분류 헤드
        self.mlp_head = MLPHead()
    
    def forward(self, x):
        x = self.patcher(x)              # 이미지 → 패치 임베딩 시퀀스
        x = self.pos_embedding(x)        # Class Token + 위치 정보 추가
        x = self.transformer_encoders(x) # Self-Attention으로 패치 간 관계 학습
        x = x[:, 0]                      # Class Token 추출 (전체 이미지 표현)
        x = self.mlp_head(x)             # 클래스별 로짓 출력
        
        return x

In [14]:
# ViT 모델 생성 및 GPU/CPU로 이동
vit = ViT().to(device)

In [15]:
# ========== 데이터셋 로드 및 전처리 ==========
transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),   # IMAGE_SIZE x IMAGE_SIZE로 리사이즈
    transforms.ToTensor(),             # (C, H, W) 텐서로 변환, 픽셀값 [0, 1] 정규화
])

# FashionMNIST: 10개 의류 클래스, 28x28 그레이
train_dataset = datasets.FashionMNIST(root='.', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='.', train=False, download=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)   # 학습용 (셔플)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)    # 평가용

100%|██████████| 26.4M/26.4M [00:07<00:00, 3.33MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 106kB/s]
100%|██████████| 4.42M/4.42M [00:04<00:00, 1.03MB/s]
100%|██████████| 5.15k/5.15k [00:00<?, ?B/s]


In [16]:
# 데이터 로더에서 배치 하나를 꺼내 입력 텐서 shape 확인
a, b = next(iter(train_loader))
a.shape

torch.Size([128, 1, 32, 32])

In [18]:
# ========== 모델 학습 ==========
# RTX 3060 : 약 3.22분 소요
# Colab : 약 4분 소요
# CPU : 약 20분 소요
optimizer = optim.AdamW(vit.parameters(), lr=0.001)  # AdamW 옵티마이저
criterion = nn.CrossEntropyLoss()                     # 다중 클래스 분류 손실 함수
epochs = 20

for epoch in range(epochs):
  training_loss = 0.0

  for data in train_loader:
    inputs, labels = data
    inputs, labels = inputs.to(device), labels.to(device)  # GPU/CPU로 이동

    optimizer.zero_grad()          # 이전 gradient 초기화

    outputs = vit(inputs)          # 순전파
    loss = criterion(outputs, labels)
    loss.backward()                # 역전파
    optimizer.step()               # 가중치 업데이트

    training_loss += loss.item()

  # 에포크별 평균 손실 출력
  print(f'Epoch {epoch + 1}/{epochs} loss: {training_loss / len(train_loader):.3f}')

Epoch 1/20 loss: 0.754
Epoch 2/20 loss: 0.478
Epoch 3/20 loss: 0.429
Epoch 4/20 loss: 0.402
Epoch 5/20 loss: 0.384
Epoch 6/20 loss: 0.368
Epoch 7/20 loss: 0.358
Epoch 8/20 loss: 0.346
Epoch 9/20 loss: 0.333
Epoch 10/20 loss: 0.327
Epoch 11/20 loss: 0.318
Epoch 12/20 loss: 0.313
Epoch 13/20 loss: 0.305
Epoch 14/20 loss: 0.300
Epoch 15/20 loss: 0.292
Epoch 16/20 loss: 0.291
Epoch 17/20 loss: 0.283
Epoch 18/20 loss: 0.280
Epoch 19/20 loss: 0.274
Epoch 20/20 loss: 0.271


In [19]:
# ========== 테스트셋 정확도 평가 ==========
correct = 0
total = 0

with torch.no_grad():    # 추론 시 gradient 계산 비활성화
  for data in test_loader:
    images, labels = data
    images, labels = images.to(device), labels.to(device)

    outputs = vit(images)

    _, predicted = torch.max(outputs.data, 1)   # 가장 높은 확률의 클래스 선택
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

  print(f'\nModel Accuracy: {100 * correct // total} %')


Model Accuracy: 87 %
